<h1 style="color:#D30982;text-align:center;">Chapter 7: Plugins &amp; Extending the QDK</h1>

<h2 style="color:#D30982;">What you'll learn</h2>

- How the QDK algorithm registry works: factories, `available()`, `create()`, `register()`
- What external plugins (PySCF, Qiskit) add to the registry at import time
- How to define custom settings with type constraints and defaults
- How to subclass an existing algorithm base class and register a new implementation
- How to build a custom active space selector using SCF orbital energies as the selection criterion

<h2 style="color:#D30982;">The plugin system</h2>

Every algorithm in the QDK is accessed through a central registry using three functions you have used throughout this course: `available()`, `create()`, and `print_settings()`. This chapter reveals what is happening behind the scenes — and shows you how to add your own algorithms to the same registry.

The registry uses a **factory pattern**: each algorithm type (e.g., `"active_space_selector"`, `"scf_solver"`) has one factory object that tracks all available implementations. When you call `create("active_space_selector", "qdk_valence")`, the factory looks up the `"qdk_valence"` implementation and returns a fresh instance. When you call `register(lambda: MySelector())`, you are adding a new implementation to the factory for your algorithm's type — making it available to `create()` like any built-in.

External packages like PySCF and Qiskit connect to this system through plugins: importing `qdk_chemistry.plugins.pyscf` triggers a registration sequence that adds PySCF-backed implementations to several factories. After the import, those implementations appear in `available()` alongside the native QDK ones.

The chapter ends with a complete, working example: an `OrbitalEnergyWindowSelector` that selects active orbitals based on how close their Fock eigenvalue is to the HOMO-LUMO midpoint — a different physical criterion from the built-in occupation-based selectors.

In [ ]:
# Setup: SCF calculation for N₂
from pathlib import Path
import numpy as np

import qdk_chemistry.plugins.pyscf
import qdk_chemistry.plugins.qiskit

from qdk_chemistry.data import Structure, Orbitals, Wavefunction, Settings, Configuration
from qdk_chemistry._core.data import SciWavefunctionContainer
from qdk_chemistry.algorithms import available, create, print_settings, register, ActiveSpaceSelector
from qdk_chemistry.algorithms import registry
from qdk_chemistry.utils import Logger, compute_valence_space_parameters

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")
structure = Structure.from_xyz_file(N2_XYZ)
E_hf, wfn_hf = create("scf_solver").run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")

n_a, n_b = wfn_hf.get_total_num_electrons()
energies_a, _ = wfn_hf.get_orbitals().get_energies()
print(f"N\u2082: {n_a + n_b} electrons, {wfn_hf.get_orbitals().get_num_molecular_orbitals()} MOs")
print(f"HOMO (index {n_a - 1}): {energies_a[n_a - 1]:.4f} Ha  "
      f"LUMO (index {n_a}): {energies_a[n_a]:.4f} Ha")
print(f"HOMO-LUMO gap: {(energies_a[n_a] - energies_a[n_a - 1]) * 27.211:.3f} eV")

<h2 style="color:#D30982;">Part 1: The Registry System</h2>

`available()` with no arguments returns a dictionary mapping every registered algorithm type to its list of implementations. `registry.show_default()` shows which implementation `create()` picks when no name is specified.

Four functions drive the registry lifecycle:
- `available()` — query what is registered
- `create()` — instantiate by type + name
- `register(generator)` — add a new Python-implemented algorithm; `generator` is a lambda called each time a new instance is needed; it is also called once immediately to discover the algorithm's `type_name()` and `name()`
- `register_factory(factory)` — add an entirely new algorithm type (needed when your algorithm doesn't fit any existing type)

In [ ]:
# Explore all registered algorithm types and implementations
print("All algorithm types and their implementations:")
for alg_type, impls in available().items():
    print(f"  {alg_type:<45} {impls}")
print()
print("Default implementations:")
for alg_type, default in registry.show_default().items():
    print(f"  {alg_type:<45} \u2192 {default!r}")
# The factory pattern: each algorithm type has one factory.
# register(generator)        → adds a new implementation to the matching factory
# register_factory(factory)  → adds an entirely new algorithm type
# create()                   → asks the factory to instantiate by name
# available()                → lists what the factory currently knows about

<h2 style="color:#D30982;">Part 2: External Plugins</h2>

Plugins are Python modules that call `register()` at import time. Importing `qdk_chemistry.plugins.pyscf` adds PySCF-backed implementations to several existing factories; importing `qdk_chemistry.plugins.qiskit` adds Qiskit-backed implementations. Once imported, their implementations are indistinguishable from native QDK ones — you create and configure them the same way.

The table below shows which implementations each plugin adds. All of these were imported in the setup cell, which is why they appear in `available()` throughout this course: `pyscf_avas`, `pyscf` (SCF), `pyscf_multi` (localizer) from the PySCF plugin; `qiskit_regular_isometry`, `qiskit_standard`, `qiskit_aer_simulator` from the Qiskit plugin.

In [ ]:
# External plugins register their implementations at import time
# The pyscf plugin adds: pyscf (scf_solver), pyscf_multi (orbital_localizer), pyscf_avas (active_space_selector)
# The qiskit plugin adds: qiskit_regular_isometry (state_prep), qiskit_standard (phase_estimation),
#                          qiskit_aer_simulator (circuit_executor)

print(f"{'Algorithm type':<38} {'Plugin source':<18} {'Implementation added'}")
print("-" * 78)
for alg_type, plugin, impl in [
    ("active_space_selector",  "pyscf plugin",  "pyscf_avas"),
    ("scf_solver",             "pyscf plugin",  "pyscf"),
    ("orbital_localizer",      "pyscf plugin",  "pyscf_multi"),
    ("state_prep",             "qiskit plugin", "qiskit_regular_isometry"),
    ("phase_estimation",       "qiskit plugin", "qiskit_standard"),
    ("circuit_executor",       "qiskit plugin", "qiskit_aer_simulator"),
]:
    present = impl in available(alg_type)
    status = "✓ loaded" if present else "✗ not loaded"
    print(f"  {alg_type:<36} {plugin:<18} {impl:<30} {status}")
print()
print("Full active_space_selector list:", available("active_space_selector"))

<h2 style="color:#D30982;">Part 3: Custom Settings</h2>

Every algorithm has a `Settings` object that exposes configurable parameters via `settings().get()` and `settings().set()`. To add settings to a custom algorithm, subclass `Settings` and call `_set_default()` once per parameter in `__init__`. The arguments are:

- **key** — the string name used in `settings().get(key)` and `settings().set(key, value)`
- **type_str** — one of `'int'`, `'double'`, `'bool'`, `'string'`, `'list[int]'`, `'list[double]'`
- **default** — the value returned before the user calls `set()`
- **description** — shown in `print_settings()` output
- **limits** *(optional)* — `(min, max)` for numeric types, or a list of allowed values for strings

The custom `Settings` subclass is attached to the algorithm in `__init__` via `self._settings = MySettings()`. After calling `register()`, `print_settings("active_space_selector", "energy_window")` will display the parameter table, identical in format to any built-in algorithm.

In [ ]:
# Custom Settings: define configurable parameters for the selector
class OrbitalEnergyWindowSettings(Settings):
    def __init__(self):
        super().__init__()
        # _set_default(key, type_str, default, description, limits)
        # type_str options: 'int', 'double', 'bool', 'string', 'list[int]', 'list[double]'
        self._set_default(
            "window_hartree", "double", 1.0,
            "Half-width of energy window around the HOMO-LUMO midpoint (Hartree)",
            (0.01, 100.0),
        )

s = OrbitalEnergyWindowSettings()
print(f"Default: window_hartree = {s.get('window_hartree')}")
s.set("window_hartree", 0.5)
print(f"After set(0.5): window_hartree = {s.get('window_hartree')}")
print()
print("The settings object is attached to the selector via self._settings.")
print("Users configure it with sel.settings().set('window_hartree', 0.5) after create().")

<h2 style="color:#D30982;">Part 4: Building and Registering the Selector</h2>

To build a custom active space selector, subclass `ActiveSpaceSelector` and implement two methods:

- `name(self) → str` — the registry key; used in `create("active_space_selector", name)` and shown by `available()`
- `_run_impl(self, wavefunction) → Wavefunction` — selection logic; receives the input wavefunction and must return a new `Wavefunction` with active space indices set on its `Orbitals`

The `OrbitalEnergyWindowSelector` below selects all molecular orbitals whose SCF orbital energy (Fock eigenvalue) lies within `±window_hartree` of the HOMO-LUMO midpoint. Unlike `qdk_occupation` (which uses fractional RDM occupations from a post-HF run), this criterion is computable directly from the HF solution — a fast heuristic for initial active space estimation.

The construction pattern in `_run_impl`: compute the midpoint between HOMO (ε[n_α−1]) and LUMO (ε[n_α]); select all orbitals within the window; identify frozen core orbitals (occupied but outside the window); build new `Orbitals` with those indices; wrap in a `Wavefunction` with the HF reference determinant.

Fill in the `name()` return value and the `register()` call in the cell below.

In [ ]:
# OrbitalEnergyWindowSelector: full implementation and registration
class OrbitalEnergyWindowSelector(ActiveSpaceSelector):
    """Selects active orbitals within an energy window around the HOMO-LUMO midpoint.

    Unlike occupation-based selectors (qdk_occupation), this criterion uses SCF orbital
    energies from the Fock matrix — selecting orbitals whose energy lies within
    ±window_hartree of the midpoint between HOMO and LUMO.
    """
    def __init__(self):
        super().__init__()
        self._settings = OrbitalEnergyWindowSettings()

    def name(self):
        return "???"  # fill in: the registry key used in create("active_space_selector", "???")

    def _run_impl(self, wavefunction):
        orbs          = wavefunction.get_orbitals()
        n_a, _        = wavefunction.get_total_num_electrons()
        n_mo          = orbs.get_num_molecular_orbitals()
        energies_a, _ = orbs.get_energies()
        window        = self.settings().get("window_hartree")

        midpoint     = (energies_a[n_a - 1] + energies_a[n_a]) / 2.0
        active_idx   = [i for i in range(n_mo) if abs(energies_a[i] - midpoint) <= window]
        inactive_idx = [i for i in range(n_a) if i not in active_idx]
        n_active_a   = sum(1 for i in active_idx if i < n_a)
        det_str      = "2" * n_active_a + "0" * (len(active_idx) - n_active_a)

        a_coeff, _  = orbs.get_coefficients()
        bs          = orbs.get_basis_set()
        new_orbs    = Orbitals(a_coeff, None, None, bs, (active_idx, inactive_idx))
        hf_config   = Configuration(det_str)
        return Wavefunction(SciWavefunctionContainer(np.array([1.0]), [hf_config], new_orbs))


# Register with the QDK registry: supply a lambda factory
# Pattern: register(lambda: MyClass())
generator = None  # TODO: replace None with lambda: OrbitalEnergyWindowSelector()
if generator is not None:
    register(generator)

print("active_space_selector implementations after registration attempt:")
print(available("active_space_selector"))
print()
sel = create("active_space_selector", "energy_window")
print(f"Created: name={sel.name()!r}  type={sel.type_name()!r}")
print_settings("active_space_selector", "energy_window")

# Question: ch07-q1

> Question not found (HTTP 404)


<h2 style="color:#D30982;">Part 5: Comparing Selectors</h2>

With `energy_window` registered, it participates in the same `create()` / `settings()` / `run()` workflow as any built-in selector. Varying `window_hartree` controls how many orbitals are included: a narrow window (0.3 Ha) captures only the few orbitals near the gap; a wider window (1.0 Ha) captures more of the occupied-virtual frontier.

The comparison below shows how the energy-window active space relates to the `qdk_valence` selection used throughout this course. For strongly correlated systems like stretched N₂, occupation-based and entropy-based selectors (Chapters 2–3) are more physically targeted; energy-window selection is useful as a validation cross-check or when post-HF data is not yet available.

In [ ]:
# Compare active space selectors on N₂: energy_window vs valence
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)

print(f"{'Selector':<30} {'setting':<22} {'n_active_orbs':>14} {'active_idx'}")
print("-" * 90)

wfn_val = create("active_space_selector", "qdk_valence",
                 num_active_electrons=num_val_e, num_active_orbitals=num_val_o).run(wfn_hf)
ai, _ = wfn_val.get_orbitals().get_active_space_indices()
print(f"  {'qdk_valence':<28} {'(10e / 8o fixed)':<22} {len(ai):>14}  {ai}")

for w in [0.35, 0.5, 1.0]:
    sel = create("active_space_selector", "energy_window")
    sel.settings().set("window_hartree", w)
    wfn_ew = sel.run(wfn_hf)
    ai, _ = wfn_ew.get_orbitals().get_active_space_indices()
    n_ae, _ = wfn_ew.get_active_num_electrons()
    print(f"  {'energy_window':<28} {f'window=±{w} Ha':<22} {len(ai):>14}  {ai}  ({n_ae}α e active)")

print()
print(f"HOMO: {energies_a[n_a-1]:.4f} Ha   LUMO: {energies_a[n_a]:.4f} Ha   "
      f"midpoint: {(energies_a[n_a-1]+energies_a[n_a])/2:.4f} Ha")
print()
print("→ Energy-window selection is a fast heuristic computable from HF alone.")
print("  For strongly correlated systems, occupation-based or entropy-based selectors")
print("  (Chapters 2–3) are more physically targeted; energy-window is useful for")
print("  quick validation or when SCI data is not yet available.")

<h2 style="color:#D30982;">Summary</h2>

In this chapter you:
- Explored the QDK algorithm registry: `available()`, `show_default()`, `create()`, and the factory pattern
- Mapped what the PySCF and Qiskit plugins add to the registry at import time
- Defined a custom `Settings` subclass with a typed, constrained `window_hartree` parameter
- Built `OrbitalEnergyWindowSelector`: a custom `ActiveSpaceSelector` that selects orbitals by Fock eigenvalue proximity to the HOMO-LUMO midpoint
- Registered it with `register(lambda: OrbitalEnergyWindowSelector())` and verified it appears in `available()`
- Compared the energy-window selection to the built-in `qdk_valence` selector across window sizes

**Key pattern (custom algorithm registration):**
```python
class MySelector(ActiveSpaceSelector):
    def __init__(self):
        super().__init__()
        self._settings = MySettings()    # custom Settings subclass

    def name(self):
        return "my_selector"             # registry key

    def _run_impl(self, wavefunction):
        ...                              # selection logic
        return new_wavefunction          # Wavefunction with active space set

register(lambda: MySelector())           # adds to the active_space_selector factory
sel = create("active_space_selector", "my_selector")
```

**End-to-end course complete:**
Structure (Ch.0–1) → HF/SCF (Ch.2) → active space (Ch.3) → qubit Hamiltonian (Ch.4) → state prep (Ch.5) → QPE (Ch.6) → **plugins & extension** (Ch.7) → custom quantum chemistry workflows on Magne